<a href="https://colab.research.google.com/github/Sahilkom/Intern_project/blob/main/Task3/Modified_RAG_Hybrid_Search_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Installing and Importing Required Dependencies

In [1]:
!pip install langchain rank_bm25 pypdf unstructured chromadb
!pip install unstructured['pdf'] unstructured
!apt-get install poppler-utils
!apt-get install -y tesseract-ocr
!apt-get install -y libtesseract-dev
!pip install pytesseract
!pip install fpdf
!pip install langchain_community

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.4).
0 upgraded, 0 newly installed, 0 to remove and 45 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  tesseract-ocr-eng tesseract-ocr-osd
The following NEW packages will be installed:
  tesseract-ocr tesseract-ocr-eng tesseract-ocr-osd
0 upgraded, 3 newly installed, 0 to remove and 45 not upgraded.
Need to get 4,816 kB of archives.
After this operation, 15.6 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-eng all 1:4.00~git30-7274cfa-1.1 [1,591 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-osd all 1:4.00~git30-7274cfa-1.1 [2,990 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr amd64 4.1.

In [2]:
from langchain.document_loaders import UnstructuredPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain.embeddings import HuggingFaceInferenceAPIEmbeddings
from langchain.llms import HuggingFaceHub

from langchain.retrievers import BM25Retriever, EnsembleRetriever

from fpdf import FPDF

import os

# Pre-Processing Data

In [7]:
import json
def load_data():
    with open("/content/TASK_3.json", 'r') as f:
        return json.load(f)

data=load_data()

In [8]:
# create chunks
data_set = []
for table in data['tables']:
    table_info=[]
    table_name=f"Table name: {table['table_name']}"
    table_description=f"Description: {table['description']}"
    table_info.append(table_name)
    table_info.append(table_description)
    title="Column name   "
    table_info.append(title)
    index=1
    for col in table['columns']:
        col_info=f" {col['name']}  ({col['description']})"
        col_info=str(index)+". "+col_info
        table_info.append(col_info)
        index+=1
    if data_set.count(table_info) <= 0:
        data_set.append(table_info)


In [9]:
for chunk in data_set:
    for info in chunk:
        print(info+"\n")
    print("\n")

Table name: products

Description: Stores product information

Column name   

1.  product_id  (Unique identifier for each product)

2.  product_name  (Name of the product)

3.  category  (Product category (e.g., Smartphone, TV))

4.  launch_date  (Launch date of the product)

5.  price  (Price of the product)

6.  manufacturer  (Manufacturer of the product)

7.  warranty_period  (Warranty period of the product)

8.  stock_quantity  (Quantity of product in stock)

9.  rating  (Customer rating of the product)

10.  dimensions  (Dimensions of the product)

11.  weight  (Weight of the product)

12.  color  (Color of the product)

13.  material  (Material of the product)

14.  power_usage  (Power usage of the product)

15.  model_number  (Model number of the product)



Table name: sales

Description: Stores sales information

Column name   

1.  sale_id  (Unique identifier for each sale)

2.  customer_id  (ID of the customer who made the purchase)

3.  product_id  (ID of the purchased pro

In [10]:
pdf = FPDF()

for table in data_set:
  pdf.add_page()
  for info in table:
    pdf.set_font("Arial", size = 10)
    pdf.cell(2000, 10, txt = info, ln = 1, align = 'L')

pdf.output("new_data.pdf")

''

# Importing processed data in the form of Documnet

In [11]:
file_path = "/content/new_data.pdf"
data_file = UnstructuredPDFLoader(file_path)
docs = data_file.load()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


In [12]:
print(docs[0].page_content)

Table name: products

Description: Stores product information

Column name

1. product_id (Unique identifier for each product)

2. product_name (Name of the product)

3. category (Product category (e.g., Smartphone, TV))

4. launch_date (Launch date of the product)

5. price (Price of the product)

6. manufacturer (Manufacturer of the product)

7. warranty_period (Warranty period of the product)

8. stock_quantity (Quantity of product in stock)

9. rating (Customer rating of the product)

10. dimensions (Dimensions of the product)

11. weight (Weight of the product)

12. color (Color of the product)

13. material (Material of the product)

14. power_usage (Power usage of the product)

15. model_number (Model number of the product)

Table name: sales

Description: Stores sales information

Column name

1. sale_id (Unique identifier for each sale)

2. customer_id (ID of the customer who made the purchase)

3. product_id (ID of the purchased product)

4. sale_date (Date of the sale)

5. q

# Importing Feature Extraction Model

In [13]:
# Get Embedding Model from HF via API

from google.colab import userdata
HF_TOKEN = userdata.get('HUGGINGFACEHUB_API_TOKEN')

embeddings = HuggingFaceInferenceAPIEmbeddings(
    api_key=HF_TOKEN, model_name="BAAI/bge-base-en-v1.5"
)

### VectorStore

In [14]:
# Vector store with the selected embedding model
vectorstore = Chroma.from_documents(docs, embeddings)

In [15]:
# vectorstore_retreiver = vectorstore.as_retriever(search_kwargs={"k": 3})
vectorstore_retreiver = vectorstore.as_retriever()

In [16]:
keyword_retriever = BM25Retriever.from_documents(docs)
# keyword_retriever.k =  3

### Ensemble Retriever

In [29]:
ensemble_retriever = EnsembleRetriever(retrievers=[vectorstore_retreiver,
                                                   keyword_retriever],
                                       weights=[0.6,0.4])

In [32]:
llm = HuggingFaceHub(
    repo_id="mistralai/Mistral-7B-Instruct-v0.3",
    model_kwargs={"temperature": 0.3,"max_new_tokens":1024},
    huggingfacehub_api_token=HF_TOKEN,
)

### Prompt Template:

In [31]:
template = """
<|system|>>
                                              !! Hello !!
                                          This is AI Model-2.0
                                        How may I help you today?

CONTEXT: {context}
</s>
<|user|>
{query}
</s>
<|assistant|>
"""

In [33]:
prompt = ChatPromptTemplate.from_template(template)
output_parser = StrOutputParser()

In [34]:
chain = (
    {"context": ensemble_retriever, "query": RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

# Queries

In [23]:
print(chain.invoke("Give me list of all tables present in data"))

Human: 
<|system|>>
                                              !! Hello !!
                                          This is AI Model-2.0
                                        How may I help you today?

CONTEXT: [Document(metadata={'source': '/content/new_data.pdf'}, page_content="Table name: products\n\nDescription: Stores product information\n\nColumn name\n\n1. product_id (Unique identifier for each product)\n\n2. product_name (Name of the product)\n\n3. category (Product category (e.g., Smartphone, TV))\n\n4. launch_date (Launch date of the product)\n\n5. price (Price of the product)\n\n6. manufacturer (Manufacturer of the product)\n\n7. warranty_period (Warranty period of the product)\n\n8. stock_quantity (Quantity of product in stock)\n\n9. rating (Customer rating of the product)\n\n10. dimensions (Dimensions of the product)\n\n11. weight (Weight of the product)\n\n12. color (Color of the product)\n\n13. material (Material of the product)\n\n14. power_usage (Power usage of t

In [35]:
print(chain.invoke("give me tables that can be join on the basis of column description"))

Human: 
<|system|>>
                                              !! Hello !!
                                          This is AI Model-2.0
                                        How may I help you today?

CONTEXT: [Document(metadata={'source': '/content/new_data.pdf'}, page_content="Table name: products\n\nDescription: Stores product information\n\nColumn name\n\n1. product_id (Unique identifier for each product)\n\n2. product_name (Name of the product)\n\n3. category (Product category (e.g., Smartphone, TV))\n\n4. launch_date (Launch date of the product)\n\n5. price (Price of the product)\n\n6. manufacturer (Manufacturer of the product)\n\n7. warranty_period (Warranty period of the product)\n\n8. stock_quantity (Quantity of product in stock)\n\n9. rating (Customer rating of the product)\n\n10. dimensions (Dimensions of the product)\n\n11. weight (Weight of the product)\n\n12. color (Color of the product)\n\n13. material (Material of the product)\n\n14. power_usage (Power usage of t

In [37]:
print(chain.invoke("Give me all 10 table with their description"))

Human: 
<|system|>>
                                              !! Hello !!
                                          This is AI Model-2.0
                                        How may I help you today?

CONTEXT: [Document(metadata={'source': '/content/new_data.pdf'}, page_content="Table name: products\n\nDescription: Stores product information\n\nColumn name\n\n1. product_id (Unique identifier for each product)\n\n2. product_name (Name of the product)\n\n3. category (Product category (e.g., Smartphone, TV))\n\n4. launch_date (Launch date of the product)\n\n5. price (Price of the product)\n\n6. manufacturer (Manufacturer of the product)\n\n7. warranty_period (Warranty period of the product)\n\n8. stock_quantity (Quantity of product in stock)\n\n9. rating (Customer rating of the product)\n\n10. dimensions (Dimensions of the product)\n\n11. weight (Weight of the product)\n\n12. color (Color of the product)\n\n13. material (Material of the product)\n\n14. power_usage (Power usage of t

In [38]:
print(chain.invoke("How many tables have a column related to price or cost?"))

Human: 
<|system|>>
                                              !! Hello !!
                                          This is AI Model-2.0
                                        How may I help you today?

CONTEXT: [Document(metadata={'source': '/content/new_data.pdf'}, page_content="Table name: products\n\nDescription: Stores product information\n\nColumn name\n\n1. product_id (Unique identifier for each product)\n\n2. product_name (Name of the product)\n\n3. category (Product category (e.g., Smartphone, TV))\n\n4. launch_date (Launch date of the product)\n\n5. price (Price of the product)\n\n6. manufacturer (Manufacturer of the product)\n\n7. warranty_period (Warranty period of the product)\n\n8. stock_quantity (Quantity of product in stock)\n\n9. rating (Customer rating of the product)\n\n10. dimensions (Dimensions of the product)\n\n11. weight (Weight of the product)\n\n12. color (Color of the product)\n\n13. material (Material of the product)\n\n14. power_usage (Power usage of t

In [39]:
print(chain.invoke("Give me a primary key for each table"))

Human: 
<|system|>>
                                              !! Hello !!
                                          This is AI Model-2.0
                                        How may I help you today?

CONTEXT: [Document(metadata={'source': '/content/new_data.pdf'}, page_content="Table name: products\n\nDescription: Stores product information\n\nColumn name\n\n1. product_id (Unique identifier for each product)\n\n2. product_name (Name of the product)\n\n3. category (Product category (e.g., Smartphone, TV))\n\n4. launch_date (Launch date of the product)\n\n5. price (Price of the product)\n\n6. manufacturer (Manufacturer of the product)\n\n7. warranty_period (Warranty period of the product)\n\n8. stock_quantity (Quantity of product in stock)\n\n9. rating (Customer rating of the product)\n\n10. dimensions (Dimensions of the product)\n\n11. weight (Weight of the product)\n\n12. color (Color of the product)\n\n13. material (Material of the product)\n\n14. power_usage (Power usage of t

In [40]:
print(chain.invoke("Give me foreign key relationship with department table and other table"))

Human: 
<|system|>>
                                              !! Hello !!
                                          This is AI Model-2.0
                                        How may I help you today?

CONTEXT: [Document(metadata={'source': '/content/new_data.pdf'}, page_content="Table name: products\n\nDescription: Stores product information\n\nColumn name\n\n1. product_id (Unique identifier for each product)\n\n2. product_name (Name of the product)\n\n3. category (Product category (e.g., Smartphone, TV))\n\n4. launch_date (Launch date of the product)\n\n5. price (Price of the product)\n\n6. manufacturer (Manufacturer of the product)\n\n7. warranty_period (Warranty period of the product)\n\n8. stock_quantity (Quantity of product in stock)\n\n9. rating (Customer rating of the product)\n\n10. dimensions (Dimensions of the product)\n\n11. weight (Weight of the product)\n\n12. color (Color of the product)\n\n13. material (Material of the product)\n\n14. power_usage (Power usage of t